In [7]:
import pandas as pd, numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, f1_score, precision_score, recall_score


df = pd.read_csv('data/combined.csv')

df['Date_x'] = pd.to_datetime(df['Date_x']).dt.date
df['daily_change_y'] = pd.to_numeric(df['daily_change_y'], errors='coerce')
df = df.dropna(subset=['daily_change_y']).copy()

if 'Surprise(%)' in df.columns:
    df["Surprise(%)"] = pd.to_numeric(df["Surprise(%)"], errors="coerce")
    df = df.sort_values(["ticker","Date_x"])
    df["surprise_shift1"] = df.groupby("ticker")["Surprise(%)"].shift(1)

# Drop leaky columns
for col in ["Reported EPS","Surprise(%)","Adj Close"]:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

df["high_low_range"] = (df["High"] - df["Low"]) / df["Open"]
if "sma20" in df.columns:
    df["close_sma20_ratio"] = df["Open"] / df["sma20"]
df["vol_log"] = np.log1p(df["Volume"].fillna(0))

df['y'] = (df['daily_change_y'] >= 0.03).astype(int)

num_feats = ['EPS Estimate', 'TTM_EPS', 'PE_Ratio', 'Forward_PE', 'Open',
             'High', 'Low', 'Volume', '1w_change', '1m_change', '3m_change',
             '1yr_change', '52wk_high', '52wk_low', 'sma20_change', 'sma50_change',
             'sma200_change', 'rsi_14', 'volatility_wk', 'volatility_1m', 
             'shifted_rep_eps', 'surprise_shift1', 'high_low_range', 'close_sma20_ratio',
             'vol_log']
num_feats = [f for f in num_feats if f in df.columns]

cat_feats = ['Quarter', 'Sector', 'Industry', 'HeadquartersState']
cat_feats = [f for f in cat_feats if f in df.columns]

if "Date_x" in df.columns and df["Date_x"].notna().any():
    df = df.sort_values("Date_x").reset_index(drop=True)
else:
    df = df.reset_index(drop=True)
split_idx = int(len(df) * 0.8)
train = df.iloc[:split_idx].copy()
test = df.iloc[split_idx:].copy()


numeric_transform = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
cat_transform = Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")), ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

preprocessor = ColumnTransformer([("num", numeric_transform, num_feats),
                                  ("cat", cat_transform, cat_feats)])

In [ ]:
def fit_pipeline(X, y):
    pipe = Pipeline([("pre", preprocessor), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))])
    pipe.fit(X, y)
    return pipe

# --- Train general model ---
X_train = train[num_feats + cat_feats]
y_train = train["y"]
X_test = test[num_feats + cat_feats]
y_test = test["y"]
general_pipe = fit_pipeline(X_train, y_train)
probs_general = general_pipe.predict_proba(X_test)[:,1]

def metrics_from_probs(y_true, probs):
    roc = roc_auc_score(y_true, probs)
    prec, rec, _ = precision_recall_curve(y_true, probs)
    pr_auc = auc(rec, prec)
    pred05 = (probs >= 0.5).astype(int)
    f1_05 = f1_score(y_true, pred05)
    grid = np.linspace(0,1,101)
    f1s = [f1_score(y_true, (probs>=g).astype(int)) for g in grid]
    best_idx = int(np.nanargmax(f1s))
    best_thr = float(grid[best_idx])
    best_f1 = float(f1s[best_idx])
    pred_best = (probs >= best_thr).astype(int)
    p_best = precision_score(y_true, pred_best, zero_division=0)
    r_best = recall_score(y_true, pred_best, zero_division=0)
    return {"roc_auc":roc, "pr_auc":pr_auc, "f1@0.5":f1_05, "best_f1":best_f1, "best_thr":best_thr, "prec@best":p_best, "rec@best":r_best}

print("General model test metrics:", metrics_from_probs(y_test.values, probs_general))

# --- Groups to evaluate ---
states = ["California", "Texas", "New York", "Illinois", "Pennsylvania", "Michigan"]
sectors = ["Financials", "Technology", "Health Care", "Business Services"]
industries = ["Specialty Retailers"]

def evaluate_group_blend(train, test, filter_col, filter_val, min_train=200, min_test=50):
    train_subset = train[train[filter_col] == filter_val].copy()
    n_train = len(train_subset)
    test_subset = test[test[filter_col] == filter_val].copy()
    n_test = len(test_subset)
    if n_train < min_train or n_test < min_test:
        return {"group": f"{filter_col}={filter_val}", "note": "insufficient_data", "n_train": n_train, "n_test": n_test}
    X_tr = train_subset[num_feats + cat_feats]; y_tr = train_subset["y"]
    spec_pipe = fit_pipeline(X_tr, y_tr)
    mask = (test[filter_col] == filter_val).values
    if not mask.any():
        return {"group": f"{filter_col}={filter_val}", "note": "no_group_in_test", "n_train": n_train, "n_test": n_test}
    probs_gen = probs_general.copy()
    X_grp = test[mask][num_feats + cat_feats]
    probs_spec_grp = spec_pipe.predict_proba(X_grp)[:,1]
    probs_spec = np.zeros_like(probs_gen); probs_spec[mask] = probs_spec_grp
    probs_blend = probs_gen.copy(); probs_blend[mask] = probs_gen[mask] * 0.3 + probs_spec[mask] * 0.7 ## (probs_gen[mask] + probs_spec[mask]) / 2.0
    return {"group": f"{filter_col}={filter_val}", "n_train": n_train, "n_test": n_test,
            "general_metrics": metrics_from_probs(y_test.values, probs_gen),
            "blended_metrics": metrics_from_probs(y_test.values, probs_blend)}

results = []
for s in states:
    if "HeadquartersState" in train.columns:
        results.append(evaluate_group_blend(train, test, "HeadquartersState", s))
for s in sectors:
    if "Sector" in train.columns:
        results.append(evaluate_group_blend(train, test, "Sector", s))
for s in industries:
    if "Industry" in train.columns:
        results.append(evaluate_group_blend(train, test, "Industry", s))

General model test metrics: {'roc_auc': 0.6284793848244112, 'pr_auc': 0.2779605507399029, 'f1@0.5': 0.38618524332810045, 'best_f1': 0.3987175750510055, 'best_thr': 0.43, 'prec@best': 0.2675009777082519, 'rec@best': 0.782608695652174}


In [9]:
# Build summary table
import warnings
warnings.filterwarnings("ignore")
rows = []
for r in results:
    if "note" in r:
        rows.append({"group": r.get("group"), "status": r.get("note"), "n_train": r.get("n_train"), "n_test": r.get("n_test")})
    else:
        gg = r["general_metrics"]; gb = r["blended_metrics"]
        rows.append({
            "group": r["group"], "status": "ok", "n_train": r["n_train"], "n_test": r["n_test"],
            "roc_general": gg["roc_auc"], "roc_blend": gb["roc_auc"],
            "pr_general": gg["pr_auc"], "pr_blend": gb["pr_auc"],
            "f1@0.5_general": gg["f1@0.5"], "f1@0.5_blend": gb["f1@0.5"],
            "best_f1_general": gg["best_f1"], "best_f1_blend": gb["best_f1"],
            "best_thr_general": gg["best_thr"], "best_thr_blend": gb["best_thr"],
            "prec@best_general": gg["prec@best"], "prec@best_blend": gb["prec@best"],
            "rec@best_general": gg["rec@best"], "rec@best_blend": gb["rec@best"]
        })

summary_df = pd.DataFrame(rows)
summary_df.to_csv("blended_model_results_summary1.csv", index=False)
print("Saved blended_model_results_summary.csv")
print(summary_df)

Saved blended_model_results_summary.csv
                             group             status  n_train  n_test  \
0     HeadquartersState=California                 ok     2077     536   
1          HeadquartersState=Texas                 ok     1687     419   
2       HeadquartersState=New York                 ok     1291     322   
3       HeadquartersState=Illinois                 ok     1037     258   
4   HeadquartersState=Pennsylvania                 ok      721     182   
5       HeadquartersState=Michigan                 ok      496     114   
6                Sector=Financials                 ok     2268     556   
7                Sector=Technology                 ok     1972     503   
8               Sector=Health Care                 ok     1343     353   
9         Sector=Business Services                 ok      956     259   
10    Industry=Specialty Retailers  insufficient_data        0       0   

    roc_general  roc_blend  pr_general  pr_blend  f1@0.5_general  \
0  